# Dogs vs Cats Image Classification

**Goal:** Classify images as Dog or Cat
**Algorithm:** Convolutional Neural Network (CNN) with Keras

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow import keras
from keras import layers
%matplotlib inline

In [ ]:
np.random.seed(42)
tf.random.set_seed(42)

# Create synthetic image data (32x32 RGB, 0=cat, 1=dog)
n_train = 500
n_test = 100
img_size = 32

# Generate synthetic images with different patterns
def generate_dog_image():
    img = np.random.randn(img_size, img_size, 3) * 0.1
    # Dogs have more horizontal edge patterns
    img[10:20, :, 0] += 0.5
    img[15:18, 10:25, 1] += 0.3
    return np.clip(img, 0, 1)

def generate_cat_image():
    img = np.random.randn(img_size, img_size, 3) * 0.1
    # Cats have more vertical edge patterns
    img[:, 10:20, 0] += 0.5
    img[10:25, 15:18, 2] += 0.3
    return np.clip(img, 0, 1)

X_dog = np.array([generate_dog_image() for _ in range(n_train // 2 + n_test // 2)])
X_cat = np.array([generate_cat_image() for _ in range(n_train // 2 + n_test // 2)])
y_dog = np.ones(len(X_dog))
y_cat = np.zeros(len(X_cat))

X = np.vstack([X_dog, X_cat])
y = np.hstack([y_dog, y_cat])

print ('Total images: %d' % len(X))
print ('Dogs: %d, Cats: %d' % (y.sum(), len(y) - y.sum()))

<hr>## 1. Explore Sample Images

In [ ]:
plt.figure(figsize=(8, 4))
for i in range(4):
    plt.subplot(1, 4, i + 1)
    idx = i * 50
    plt.imshow(X[idx])
    plt.title('Dog' if y[idx] == 1 else 'Cat')
    plt.axis('off')
plt.tight_layout()
plt.show()

<hr>## 2. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print ('Train: %d, Test: %d' % (len(X_train), len(X_test)))
print ('Image shape: %s' % (X_train.shape[1:],))

<hr>## 3. Build CNN Model

In [ ]:
model = keras.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)),
    layers.MaxPooling2D(2, 2),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),
    layers.Flatten(),
    layers.Dropout(0.5),
    layers.Dense(512, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)
print ('CNN Architecture:')
model.summary()

<hr>## 4. Train the CNN

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=10,
    batch_size=32,
    verbose=1
)

<hr>## 5. Evaluate Performance

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print ('Test Accuracy: %.4f (%.1f%%)' % (test_acc, test_acc * 100))
print ('Test Loss: %.4f' % test_loss)

In [ ]:
y_pred = (model.predict(X_test) > 0.5).astype(int).flatten()
print ('Classification Report:\n%s' % classification_report(
    y_test, y_pred, target_names=['Cat', 'Dog']))

<hr>## 6. Plot Training History

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Validation')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Model Accuracy')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Validation')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Model Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
print ('Final validation accuracy: %.2f%%' % (history.history['val_accuracy'][-1] * 100))
print ('Final validation loss: %.4f' % history.history['val_loss'][-1])